# RAG

Un LLM no conoce **tus** documentos. **RAG** (Retrieval-Augmented Generation) lo soluciona: antes de responder, **busca** la información relevante en tus textos y se la **pasa** al modelo.


## Configuración

Instalamos lo necesario:
- **`openai`** → para hablar con el modelo (Groq u OpenRouter), como siempre.
- **`sentence-transformers`** → para crear los embeddings **en local**.
- **`numpy`** → para las cuentas de la búsqueda.

> La primera vez, `sentence-transformers` descarga un modelo pequeño (~90 MB). Tarda un poco solo la primera vez.

In [ ]:
# !pip install -q sentence-transformers

In [1]:
import numpy as np
from openai import OpenAI
from getpass import getpass

In [2]:
API_KEY = getpass("Pega tu clave de Groq u OpenRouter: ")

BASE_URL = "https://api.groq.com/openai/v1"
MODELO   = "llama-3.3-70b-versatile"

cliente = OpenAI(api_key=API_KEY, base_url=BASE_URL)

## La base de conocimiento

Estos son "nuestros documentos": datos de **The Bridge** que el modelo **no puede conocer** (son inventados). Cada texto es un fragmento corto sobre un tema. 

In [3]:
documentos = [
    "En The Bridge fue fundada en 2019 y está especializada en formación de Data Science e Inteligencia Artificial.",
    "El bootcamp de Data Science de The Bridge dura 14 semanas, con clases de lunes a viernes de 9:00 a 14:00.",
    "Para obtener el diploma, el alumno debe asistir al menos al 80% de las clases y entregar el proyecto final.",
    "El proyecto final del bootcamp se realiza durante las 2 últimas semanas y se presenta ante un tribunal de profesores.",
    "El plan de estudios de Data Science incluye módulos avanzados de Python, Machine Learning, Deep Learning y SQL.",
    "Los alumnos tienen acceso a un servicio de Career Readiness que les ayuda a preparar su currículum y simulacros de entrevistas técnicas.",
    "Además del formato presencial, la escuela ofrece la modalidad Full Time en remoto con clases en streaming en directo.",
    "No se requieren conocimientos previos en programación para matricularse, ya que las 2 primeras semanas se realiza un módulo introductorio de Ramp-up.",
    "La escuela cuenta con una red de más de 100 empresas colaboradoras donde los graduados realizan procesos de selección prioritarios."
]
print(f"Tenemos {len(documentos)} fragmentos en nuestra base.")

Tenemos 9 fragmentos en nuestra base.


## Embeddings: convertir texto en números

Un **embedding** transforma un texto en una lista de números (un **vector**) que captura su **significado**. La clave: **textos parecidos → vectores cercanos**. Eso es lo que nos permitirá "buscar por significado".

https://huggingface.co/spaces/hesamation/primer-llm-embedding?section=what_makes_a_good_embedding%3F

https://huggingface.co/spaces/webml-community/semantic-galaxy

Cargamos un modelo de embeddings pequeño y rápido.

In [4]:
from sentence_transformers import SentenceTransformer

# Modelo de embeddings ligero (se descarga la primera vez)
modelo_emb = SentenceTransformer("all-MiniLM-L6-v2")

ejemplo_vec = modelo_emb.encode("Hola mundo")
print("Un embedding es un vector de", len(ejemplo_vec), "números.")
print("Primeros 8 valores:", ejemplo_vec[:8])

c:\Users\Diego Nuñez\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Un embedding es un vector de 384 números.
Primeros 8 valores: [-0.04199903  0.13960984  0.00764838 -0.00863832 -0.03008484 -0.03793447
  0.08223388 -0.04104556]


## Indexamos nuestra base de conocimiento

"Indexar" = convertir **todos** los documentos en embeddings y guardarlos. Lo hacemos una sola vez. `normalize_embeddings=True` nos facilitará la búsqueda después.

In [5]:
# Convertimos los 8 documentos en una matriz de vectores
vectores = modelo_emb.encode(documentos, normalize_embeddings=True)

print("Hemos indexado", vectores.shape[0], "documentos.")
print("Cada uno es un vector de", vectores.shape[1], "números.")

Hemos indexado 9 documentos.
Cada uno es un vector de 384 números.


In [7]:
len(vectores)

9

## 4. Buscar: encontrar los fragmentos relevantes

Esta es la **R** de RAG (Retrieval = recuperación). Para una pregunta:
1. La convertimos en embedding.
2. La comparamos con **todos** los documentos.
3. Devolvemos los **más parecidos** (los `k` mejores).

Como los vectores están normalizados, comparar = un simple producto (`@` en numpy).

In [8]:
def buscar(pregunta, k=3):
    """Devuelve los k fragmentos más relevantes para la pregunta, con su puntuación."""
    v_pregunta = modelo_emb.encode(pregunta, normalize_embeddings=True)
    similitudes = vectores @ v_pregunta            # un número por documento
    mejores = np.argsort(similitudes)[::-1][:k]    # las posiciones de mayor a menor
    return [(documentos[i], float(similitudes[i])) for i in mejores]


for texto, score in buscar("¿cuánto dura el bootcamp?"):
    print(f"[{score:.3f}] {texto}")

[0.706] El proyecto final del bootcamp se realiza durante las 2 últimas semanas y se presenta ante un tribunal de profesores.
[0.536] El bootcamp de Data Science de The Bridge dura 14 semanas, con clases de lunes a viernes de 9:00 a 14:00.
[0.444] Además del formato presencial, la escuela ofrece la modalidad Full Time en remoto con clases en streaming en directo.


## 5. Generar: el RAG completo

Ahora la **G** (Generation). Juntamos todo:
1. **Buscamos** los fragmentos relevantes.
2. Los metemos en el prompt como **contexto**.
3. Le pedimos al modelo que responda **usando solo ese contexto** y que **cite las fuentes**.

Esa instrucción ("usa solo el contexto") es clave para **reducir alucinaciones**.

In [9]:
def rag(pregunta, k=3, verbose=True):
    
    encontrados = buscar(pregunta, k)

    # 2) Construir el contexto con fuentes numeradas
    contexto = "\n".join(f"[Fuente {i+1}] {txt}" for i, (txt, _) in enumerate(encontrados))

    if verbose:
        print("Contexto recuperado:")
        print(contexto, "\n")

    
    prompt = f"""Responde a la pregunta usando ÚNICAMENTE la información del contexto.
Si la respuesta no está en el contexto, di claramente que no dispones de esa información.
Indica entre paréntesis qué fuente(s) has usado.

CONTEXTO:
{contexto}

PREGUNTA: {pregunta}"""

    respuesta = cliente.chat.completions.create(
        model=MODELO,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return respuesta.choices[0].message.content

# ¡Probamos nuestro RAG!
print(rag("¿Cuánto dura el bootcamp de data science y que modulos tiene?"))

Contexto recuperado:
[Fuente 1] El bootcamp de Data Science de The Bridge dura 14 semanas, con clases de lunes a viernes de 9:00 a 14:00.
[Fuente 2] El proyecto final del bootcamp se realiza durante las 2 últimas semanas y se presenta ante un tribunal de profesores.
[Fuente 3] El plan de estudios de Data Science incluye módulos avanzados de Python, Machine Learning, Deep Learning y SQL. 

El bootcamp de Data Science dura 14 semanas y tiene módulos avanzados de Python, Machine Learning, Deep Learning y SQL. (Fuente 1, Fuente 3)


## SIN RAG vs CON RAG

Hagamos la misma pregunta de dos formas. Primero **sin RAG** (el modelo solo, sin nuestros documentos): no puede saberlo, así que admitirá que no lo sabe… o se lo inventará.

In [10]:
pregunta = "¿Que formatos tiene el bootcamp?"

# --- SIN RAG: el modelo a secas ---
sin_rag = cliente.chat.completions.create(
    model=MODELO, messages=[{"role": "user", "content": pregunta}], temperature=0
).choices[0].message.content

print("SIN RAG:\n", sin_rag)

SIN RAG:
 Un bootcamp puede tener varios formatos, dependiendo de los objetivos, la duración y el enfoque del programa. A continuación, te presento algunos de los formatos más comunes:

1. **Presencial**: Los participantes se reúnen en un lugar físico, como una oficina o un centro de formación, para recibir instrucción y trabajar en proyectos.
2. **En línea**: Los participantes se conectan a través de plataformas de aprendizaje en línea, como videoconferencias, foros de discusión y recursos en línea, para recibir instrucción y trabajar en proyectos.
3. **Híbrido**: Combina elementos de los formatos presenciales y en línea. Los participantes pueden asistir a sesiones presenciales y también tener acceso a recursos en línea.
4. **Intensivo**: Un formato de bootcamp que se lleva a cabo durante un período corto de tiempo, generalmente entre 1-3 meses, con un enfoque intenso en el aprendizaje y la práctica.
5. **Part-time**: Un formato de bootcamp que se lleva a cabo durante un período más l

In [11]:
# --- CON RAG: el modelo con nuestros documentos ---
print("CON RAG:\n", rag(pregunta, verbose=False))

CON RAG:
 El bootcamp tiene dos formatos: presencial y Full Time en remoto con clases en streaming en directo. (Fuente 3)


La diferencia es enorme: **sin RAG** el modelo no conoce The Bridge; **con RAG** responde con el dato exacto (reembolso completo en los primeros 7 días) y cita la fuente. **Eso es darle conocimiento a la IA.**

## 7. Comprobamos que NO se inventa cosas

Un buen RAG también sabe decir "no lo sé" cuando la respuesta no está en los documentos. Probemos con algo que no hemos incluido.

In [12]:
print(rag("¿La academia The Bridge tiene una sede en Tokio?", verbose=False))

No dispongo de esa información. (No se menciona en ninguna de las fuentes proporcionadas: Fuente 1, Fuente 2, Fuente 3)
